## Dummy Data Generator function

In [2]:
import numpy as np
import pandas as pd

def generate_customer_data(n=300, noise=0.1, random_state=42):
    rng = np.random.default_rng(random_state)

    # Numeric features
    age = rng.integers(18, 60, n)
    income = rng.integers(20, 120, n)

    # Categorical features
    education = rng.choice(
        ["high_school", "bachelor", "master", "phd"],
        size=n,
        p=[0.35, 0.4, 0.2, 0.05]
    )

    city = rng.choice(
        ["Kathmandu", "Pokhara", "Biratnagar", "Lalitpur"],
        size=n,
        p=[0.4, 0.25, 0.2, 0.15]
    )

    employment = rng.choice(
        ["student", "private", "government", "self_employed"],
        size=n,
        p=[0.25, 0.35, 0.2, 0.2]
    )

    # --- Target logic (rule-based) ---
    y = (
        (age < 35) &
        (income > 60) &
        np.isin(education, ["bachelor", "master", "phd"])
    ).astype(int)

    # Boost probability based on education
    edu_bonus = {
        "high_school": 0.0,
        "bachelor": 0.05,
        "master": 0.10,
        "phd": 0.15
    }

    prob = y.astype(float)
    for k, v in edu_bonus.items():
        prob += (education == k) * v

    y = (rng.random(n) < prob).astype(int)

    # Inject label noise
    flip = rng.random(n) < noise
    y[flip] = 1 - y[flip]

    return pd.DataFrame({
        "age": age,
        "income": income,
        "education": education,
        "city": city,
        "employment": employment,
        "target": y
    })

In [6]:
df = generate_customer_data(n=1200, noise=0.25)

In [9]:
df.to_csv('customer_data.csv', index=False) # index=False -> avoids saving the DataFrame index as a column